In [23]:
import lightgbm as lgb
from lightgbm import LGBMClassifier
import numpy as np
import pandas as pd
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit

SEEDS = [42, 7, 21, 84, 108, 2026]
MIN_RECALL = 0.52
ML_CSV_PATH = r"C:\Aniket\Hosp model\Final_diabetic_data.csv"

LGB_PARAMS = {
    "n_estimators": 800,
    "learning_rate": 0.017268689827900557,
    "num_leaves": 55,
    "max_depth": 8,
    "min_child_samples": 72,
    "subsample": 0.7826262456580011,
    "subsample_freq": 1,
    "colsample_bytree": 0.7505188431637575,
    "reg_alpha": 0.027449208711527943,
    "reg_lambda": 2.2718670804240984,
    "n_jobs": -1,
    "verbose": -1,
}


def best_threshold_for_recall(y_true, y_prob, min_recall=0.50):
  prec, rec, thresh = precision_recall_curve(y_true, y_prob)
  p, r, t = prec[:-1], rec[:-1], thresh
  mask = r >= min_recall
  if not np.any(mask):
    idx = int(np.argmax(r))
    return float(t[idx]), float(p[idx]), float(r[idx]), False
  cand_p, cand_r, cand_t = p[mask], r[mask], t[mask]
  best_idx = int(np.lexsort((cand_r, cand_p))[-1])
  return (
      float(cand_t[best_idx]),
      float(cand_p[best_idx]),
      float(cand_r[best_idx]),
      True,
  )


def eval_row(model_name, seed, y_true, y_prob, threshold, best_iter=None):
  y_pred = (y_prob >= threshold).astype(int)
  tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
  row = {
      "Seed": seed,
      "Model": model_name,
      "Threshold": threshold,
      "ROC_AUC": roc_auc_score(y_true, y_prob),
      "PR_AUC": average_precision_score(y_true, y_prob),
      "Precision": precision_score(y_true, y_pred, zero_division=0),
      "Recall": recall_score(y_true, y_pred, zero_division=0),
      "TP": tp,
      "FP": fp,
      "TN": tn,
      "FN": fn,
  }
  if best_iter is not None:
    row["Best_Iter"] = int(best_iter)
  return row


df_ml = pd.read_csv(ML_CSV_PATH, dtype={"payer_code": str})
feature_cols = [
    c
    for c in df_ml.columns
    if c not in ["encounter_id", "patient_nbr", "target_lt30"]
]
X_all = df_ml[feature_cols].copy()
y_all = df_ml["target_lt30"].astype(int)
groups_all = df_ml["patient_nbr"]

num_cols = X_all.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X_all.columns if c not in num_cols]

for c in cat_cols:
  cats = sorted(X_all[c].astype(str).fillna("Missing").unique())
  X_all[c] = (
      X_all[c].astype(str).fillna("Missing").astype(pd.CategoricalDtype(categories=cats))
  )

lgb_rows = []
for seed in SEEDS:
  tv_idx, _ = next(
      GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=seed).split(
          X_all, y_all, groups=groups_all
      )
  )
  X_tv, y_tv, g_tv = (
      X_all.iloc[tv_idx],
      y_all.iloc[tv_idx],
      groups_all.iloc[tv_idx],
  )
  tr_idx, va_idx = next(
      GroupShuffleSplit(
          n_splits=1, test_size=0.15 / 0.85, random_state=seed
      ).split(X_tv, y_tv, groups=g_tv)
  )

  X_tr, y_tr = X_tv.iloc[tr_idx], y_tv.iloc[tr_idx]
  X_va, y_va = X_tv.iloc[va_idx], y_tv.iloc[va_idx]

  lgb_model = LGBMClassifier(**LGB_PARAMS, random_state=seed)
  lgb_model.fit(
      X_tr,
      y_tr,
      eval_X=X_va,
      eval_y=y_va,
      callbacks=[lgb.early_stopping(50, verbose=False)],
  )
  val_probs = lgb_model.predict_proba(X_va)[:, 1]
  thr, _, _, _ = best_threshold_for_recall(
      y_va.values, val_probs, min_recall=MIN_RECALL
  )
  lgb_rows.append(
      eval_row(
          "LightGBM",
          seed,
          y_va.values,
          val_probs,
          thr,
          lgb_model.best_iteration_ or LGB_PARAMS["n_estimators"],
      )
  )

avg_lgb_tbl = (
    pd.DataFrame(lgb_rows)
    .groupby("Model", as_index=False)[
        [
            "Best_Iter",
            "Threshold",
            "ROC_AUC",
            "PR_AUC",
            "Precision",
            "Recall",
            "TP",
            "FP",
            "TN",
            "FN",
        ]
    ]
    .mean()
)
for col in ["Threshold", "ROC_AUC", "PR_AUC", "Precision", "Recall"]:
  avg_lgb_tbl[col] = avg_lgb_tbl[col].round(4)
for col in ["Best_Iter", "TP", "FP", "TN", "FN"]:
  avg_lgb_tbl[col] = avg_lgb_tbl[col].round(1)

print(avg_lgb_tbl.to_string(index=False))

   Model  Best_Iter  Threshold  ROC_AUC  PR_AUC  Precision  Recall    TP     FP     TN    FN
LightGBM      369.3     0.1304   0.6792  0.2362     0.2021  0.5214 885.5 3499.8 9735.0 812.8


In [24]:
import lightgbm as lgb
from lightgbm import LGBMClassifier
import optuna
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

optuna.logging.set_verbosity(optuna.logging.WARNING)

OPTUNA_MIN_RECALL = 0.52
N_TRIALS = 25

optuna_rows = []
last_params = {}
for seed in SEEDS:
    tv_idx, _ = next(GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=seed).split(X_all, y_all, groups=groups_all))
    X_tv, y_tv, g_tv = X_all.iloc[tv_idx], y_all.iloc[tv_idx], groups_all.iloc[tv_idx]
    tr_idx, va_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.15 / 0.85, random_state=seed).split(X_tv, y_tv, groups=g_tv))

    X_tr, y_tr = X_tv.iloc[tr_idx], y_tv.iloc[tr_idx]
    X_va, y_va = X_tv.iloc[va_idx], y_tv.iloc[va_idx]

    def lgb_objective(trial):
        params = {
            "n_estimators": 800,
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 15, 127),
            "max_depth": trial.suggest_int("max_depth", 3, 12),
            "min_child_samples": trial.suggest_int("min_child_samples", 20, 300),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "subsample_freq": 1,
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
            "random_state": seed,
            "n_jobs": -1,
            "verbose": -1,
        }
        model = LGBMClassifier(**params)
        model.fit(X_tr, y_tr, eval_X=X_va, eval_y=y_va, callbacks=[lgb.early_stopping(50, verbose=False)])
        probs = model.predict_proba(X_va)[:, 1]
        thr, prec, rec, met = best_threshold_for_recall(y_va.values, probs, min_recall=OPTUNA_MIN_RECALL)
        trial.set_user_attr("thr", thr)
        return prec if met else rec - 1.0

    study_lgb = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=seed))
    study_lgb.optimize(lgb_objective, n_trials=N_TRIALS)
    bp_lgb = study_lgb.best_params
    thr_lgb = study_lgb.best_trial.user_attrs["thr"]

    best_lgb = LGBMClassifier(n_estimators=800, **bp_lgb, subsample_freq=1, random_state=seed, n_jobs=-1, verbose=-1)
    best_lgb.fit(X_tr, y_tr, eval_X=X_va, eval_y=y_va, callbacks=[lgb.early_stopping(50, verbose=False)])
    lgb_probs = best_lgb.predict_proba(X_va)[:, 1]
    optuna_rows.append(eval_row("Optuna LightGBM", seed, y_va.values, lgb_probs, thr_lgb))
    last_params = bp_lgb

optuna_avg_tbl = pd.DataFrame(optuna_rows).groupby("Model", as_index=False)[
    ["Threshold", "ROC_AUC", "PR_AUC", "Precision", "Recall", "TP", "FP", "TN", "FN"]
].mean()
for col in ["Threshold", "ROC_AUC", "PR_AUC", "Precision", "Recall"]:
    optuna_avg_tbl[col] = optuna_avg_tbl[col].round(4)
for col in ["TP", "FP", "TN", "FN"]:
    optuna_avg_tbl[col] = optuna_avg_tbl[col].round(1)

print(optuna_avg_tbl.to_string(index=False))
print(last_params)

          Model  Threshold  ROC_AUC  PR_AUC  Precision  Recall    TP     FP     TN    FN
Optuna LightGBM     0.1308   0.6796  0.2372     0.2038   0.521 884.8 3463.0 9771.8 813.5
{'learning_rate': 0.015281582101906117, 'num_leaves': 73, 'max_depth': 8, 'min_child_samples': 45, 'subsample': 0.8786477959398861, 'colsample_bytree': 0.6590272180121814, 'reg_alpha': 0.17248189598787408, 'reg_lambda': 0.03535265698065148}
